# 05 — Unified Generator Benchmark

This validation-only notebook answers RQ1. KID/FID use all positive validation references versus the canonical 1,361-image synthetic pool; the PRDC point estimate and repeated 80% stability subsets are balanced. Every sampled ID is recorded. The test split is forbidden.

## 1. Protocol configuration

In [ ]:
from pathlib import Path
import json
import numpy as np
import sys
ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'configs').is_dir())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'notebooks/utility'))
from notebooks.utility.generator_benchmark import (
    BENCHMARK_ROOT, CANONICAL_OUTPUTS, FEATURE_SPACES, REPRESENTATIONS,
    build_canonical_generator_provenance, build_synthetic_duplication_rows, build_train_memorization_rows,
    build_validation_similarity_rows, deterministic_sample, audit_runtime_generator_assets, diversity_metrics, efficiency_from_manifest,
    canonical_samples_from_manifest, detect_duplicate_generator_identities, eligibility_failures, evaluation_subset_size, extract_features, get_or_extract_embeddings,
    inception_v3_identity, rad_dino_identity,
    load_protocol, load_registry, metadata_positive_paths, plot_generator_summary,
    paired_kid_differences, rank_generator_family, render_similarity_panel, repeated_distribution_metrics,
    save_resampling_plan, technical_validity_row, training_corpus_from_manifest, write_csv_rows, balanced_subsample_indices,
)
protocol = load_protocol(ROOT)
registry = load_registry(ROOT)
BUILD_CANONICAL_PROVENANCE = False
RUN_REAL_BENCHMARK = False
DIVERSITY_PAIR_COUNT = 256
OUTPUT_ROOT = ROOT / BENCHMARK_ROOT
provenance_build_audit = []
if BUILD_CANONICAL_PROVENANCE:
    source_config = json.loads((ROOT / 'configs/generator_provenance_sources.json').read_text())
    shared = {'training_metadata': source_config['training_corpus_evidence'], 'training_dataset_identifier': source_config['training_dataset_identifier'], 'training_corpus_manifest': source_config['shared_training_corpus']['manifest']}
    for source in source_config['generators']:
        try:
            built = build_canonical_generator_provenance(ROOT, {**shared, **source})
            provenance_build_audit.append({'generator_id': source['generator_id'], 'status': 'built', 'lineage': built['lineage_status']})
        except Exception as exc:
            provenance_build_audit.append({'generator_id': source['generator_id'], 'status': 'refused', 'reason': f'{type(exc).__name__}: {exc}'})
{'build_canonical_provenance': BUILD_CANONICAL_PROVENANCE, 'provenance_audit': provenance_build_audit, 'run_real_benchmark': RUN_REAL_BENCHMARK, 'protocol': protocol, 'expected_outputs': CANONICAL_OUTPUTS}

## 2. Candidate discovery

In [ ]:
candidate_audits = detect_duplicate_generator_identities([audit_runtime_generator_assets(ROOT, entry, protocol) for entry in registry['generators'] if entry.get('benchmark', {}).get('enabled', False)])
candidate_audits

## 3. Candidate eligibility

In [ ]:
eligible = [row for row in candidate_audits if row['eligible_for_benchmark_execution']]
if RUN_REAL_BENCHMARK and not eligible:
    raise RuntimeError('No candidate has complete content-aware provenance, RAW/FILTERED lineage, filter provenance and a declared training corpus.')
[(row['generator_id'], row['candidate_role'], row['eligible_for_downstream_selection'], row['blockers']) for row in candidate_audits]

## 4. RAW/FILTERED image counts

In [ ]:
counts = [{
    'generator_id': row['generator_id'],
    **{name: row['representations'][name]['count'] for name in REPRESENTATIONS}
} for row in candidate_audits]
counts

## 5. Real reference set

In [ ]:
validation_paths = validation_ids = None
if RUN_REAL_BENCHMARK:
    required_metadata = [ROOT / 'data/processed/metadata/val.csv']
    missing = [str(path) for path in required_metadata if not path.is_file()]
    if missing:
        raise FileNotFoundError(f'Required benchmark metadata is missing: {missing}')
    validation_paths, validation_ids = metadata_positive_paths(ROOT, 'data/processed/metadata/val.csv')
    real_reference_count = len(validation_paths)
    reference_status = {'real_reference_count': real_reference_count, 'memorization_policy': 'generator-specific declared full training corpus'}
else:
    reference_status = {'status': 'Deferred: no real dataset was opened', 'policy': 'all validation positives; generator-specific declared full training corpus for memorization'}
reference_status

## 6. Technical validity

In [ ]:
technical_rows = None
if RUN_REAL_BENCHMARK:
    technical_rows = list()
    for audit in candidate_audits:
        if not audit['eligible_for_descriptive_benchmark']:
            continue
        entry = next(item for item in registry['generators'] if item['id'] == audit['generator_id'])
        for representation in REPRESENTATIONS:
            provenance = json.loads((ROOT / entry['provenance_manifest']).read_text())
            paths, _, _ = canonical_samples_from_manifest(ROOT, provenance[f'{representation}_sample_manifest'])
            technical_rows.append(technical_validity_row(audit['generator_id'], representation, paths, minimum_unique=protocol['synthetic_pool_target']))
    write_csv_rows(OUTPUT_ROOT / 'candidate_audit.csv', [{'generator_id': row['generator_id'], 'family': row['scientific_family'], 'role': row['candidate_role'], 'parent_generator_id': row.get('parent_generator_id'), 'model_identity_sha256': row.get('model_identity_sha256'), 'generation_identity_sha256': row.get('generation_identity_sha256'), 'duplicate_model_group': row.get('duplicate_model_group', ''), 'distinct_generator_for_ranking': row.get('distinct_generator_for_ranking', False), 'audit_mode': row['audit_mode'], 'audit_generated_at': __import__('datetime').datetime.now(__import__('datetime').timezone.utc).isoformat(), 'project_root_independent_paths': True, 'provenance_recorded': row['provenance_recorded'], 'provenance_record_schema_valid': row['provenance_record_schema_valid'], 'provenance_index_consistent': row['provenance_index_consistent'], 'runtime_manifest_hashes_declared': row['runtime_manifest_hashes_declared'], 'runtime_manifest_contents_verified': row['runtime_manifest_contents_verified'], 'runtime_assets_verified': row['runtime_assets_verified'], 'runtime_assets_unavailable': row['runtime_assets_unavailable'], 'runtime_assets_mismatch': row['runtime_assets_mismatch'], 'eligible_for_descriptive_benchmark': row['eligible_for_descriptive_benchmark'], 'eligible_for_official_family_ranking': row['eligible_for_official_family_ranking'], 'raw_count': row['representations']['raw']['count'], 'filtered_count': row['representations']['filtered']['count'], 'provenance_manifest_exists': row['provenance_manifest_exists'], 'provenance_manifest_valid': row['provenance_manifest_valid'], 'lineage_complete': row['lineage_complete'], 'raw_manifest_valid': row['raw_manifest_valid'], 'filter_manifest_valid': row['filter_manifest_valid'], 'sample_set_matches_manifest': row['sample_set_matches_manifest'], 'training_corpus_manifest': row['training_corpus_manifest'], 'training_corpus_manifest_valid': row['training_corpus_manifest_valid'], 'filter_acceptance_rate_descriptive': row.get('filter_acceptance_rate'), 'filter_provenance_complete': row.get('filter_provenance_complete'), 'provenance_failure_reason': row['provenance_failure_reason'], 'block_reasons': '; '.join(row['blockers'])} for row in candidate_audits])
    write_csv_rows(OUTPUT_ROOT / 'technical_validity.csv', technical_rows)
technical_rows if technical_rows is not None else 'Deferred until RUN_REAL_BENCHMARK=True'

## 7. Feature extraction

In [ ]:
embedding_cache_root = OUTPUT_ROOT / 'embedding_cache'
INCEPTION_CHECKPOINT_PATH = None  # local torchvision Inception_V3_Weights.IMAGENET1K_V1 file
RAD_DINO_SNAPSHOT_PATH = None     # local microsoft/rad-dino HF snapshot; downloads remain forbidden
ENCODER_IDENTITIES = {}
candidate_features = reference_features = candidate_paths = candidate_ids = None
if RUN_REAL_BENCHMARK:
    if INCEPTION_CHECKPOINT_PATH is None or RAD_DINO_SNAPSHOT_PATH is None:
        raise RuntimeError('Set both local encoder paths; the benchmark will not download models or use generic weight identifiers.')
    import torchvision
    inception_path, rad_path = Path(INCEPTION_CHECKPOINT_PATH), Path(RAD_DINO_SNAPSHOT_PATH)
    ENCODER_IDENTITIES['inception_v3'] = inception_v3_identity(torchvision.__version__, 'Inception_V3_Weights.IMAGENET1K_V1', inception_path, {'resize': 299, 'normalization': 'weights-enum-default'})
    ENCODER_IDENTITIES['rad_dino'] = rad_dino_identity('microsoft/rad-dino', rad_path, commit_hash=rad_path.name if rad_path.parent.name == 'snapshots' else None, config_path=rad_path / 'config.json', weight_paths=sorted(rad_path.glob('*.safetensors')), processor_config_path=rad_path / 'preprocessor_config.json', preprocessing_configuration={'processor': 'local AutoImageProcessor configuration'})
    candidate_features, reference_features, candidate_paths, candidate_ids = dict(), dict(), dict(), dict()
    execution_ids = {row['generator_id'] for row in candidate_audits if row['eligible_for_benchmark_execution']}
    benchmark_keys = {(row['generator_id'], row['condition'].lower()) for row in technical_rows if row['eligible_for_distribution_metrics'] and row['generator_id'] in execution_ids}
    reference_sets = {'validation': (validation_paths, validation_ids, 'data/processed/metadata/val.csv')}
    for pool_name, (paths, ids, manifest) in reference_sets.items():
        for extractor_name in FEATURE_SPACES:
            cache = embedding_cache_root / '_references' / pool_name / f'{extractor_name}.npy'
            reference_features[(pool_name, extractor_name)], _ = get_or_extract_embeddings(
                cache, paths, ids, extractor=extractor_name, preprocessing='registered frozen extractor preprocessing',
                code_version='embedding-integrity-v3', source_manifest=str(ROOT / manifest), metadata_csv=str(ROOT / manifest),
                extractor_model_id=ENCODER_IDENTITIES[extractor_name].get('model_repository', ENCODER_IDENTITIES[extractor_name].get('weights_enum')), extractor_weights_identifier=ENCODER_IDENTITIES[extractor_name]['identity_sha256'], extractor_identity=ENCODER_IDENTITIES[extractor_name],
                extract_fn=lambda values, name: extract_features(values, name, allow_model_download=False))
    for audit in candidate_audits:
        entry = next(item for item in registry['generators'] if item['id'] == audit['generator_id'])
        for representation in REPRESENTATIONS:
            key = (audit['generator_id'], representation)
            if key not in benchmark_keys:
                continue
            provenance = json.loads((ROOT / entry['provenance_manifest']).read_text())
            discovered_paths, discovered_ids, _ = canonical_samples_from_manifest(ROOT, provenance[f'{representation}_sample_manifest'])
            selected = set(deterministic_sample(discovered_paths, protocol['synthetic_pool_target'], protocol['sampling']['seed']))
            pairs = [(path, sample_id) for path, sample_id in zip(discovered_paths, discovered_ids) if str(path) in selected]
            paths, ids = [item[0] for item in pairs], [item[1] for item in pairs]; candidate_paths[key] = paths; candidate_ids[key] = ids
            for extractor_name in FEATURE_SPACES:
                cache = embedding_cache_root / audit['generator_id'] / representation / f'{extractor_name}.npy'
                candidate_features[(audit['generator_id'], representation, extractor_name)], _ = get_or_extract_embeddings(
                    cache, paths, ids, extractor=extractor_name, preprocessing='registered frozen extractor preprocessing',
                    code_version='embedding-integrity-v3', source_manifest=str(ROOT / entry['provenance_manifest']),
                    extractor_model_id=ENCODER_IDENTITIES[extractor_name].get('model_repository', ENCODER_IDENTITIES[extractor_name].get('weights_enum')), extractor_weights_identifier=ENCODER_IDENTITIES[extractor_name]['identity_sha256'], extractor_identity=ENCODER_IDENTITIES[extractor_name],
                    extract_fn=lambda values, name: extract_features(values, name, allow_model_download=False))
feature_status = 'Deferred' if candidate_features is None else {'candidate_feature_sets': len(candidate_features), 'reference_feature_sets': len(reference_features)}
feature_status

## 8. Distribution metrics

In [ ]:
distribution_summaries = distribution_repetitions = None
if RUN_REAL_BENCHMARK:
    distribution_summaries, distribution_repetitions = list(), list()
    stability_size = evaluation_subset_size(protocol['synthetic_pool_target'], len(validation_ids), protocol['synthetic_pool_target'], protocol['resampling']['subsampling_fraction'])
    shared_resampling_plan = balanced_subsample_indices(len(validation_ids), protocol['synthetic_pool_target'], stability_size, protocol['resampling']['stability_repetitions'], protocol['sampling']['seed'], nearest_neighbour_k=protocol['resampling']['nearest_neighbour_k'])
    save_resampling_plan(OUTPUT_ROOT / 'resampling_plan.json', shared_resampling_plan, protocol)
    eligibility = {(row['generator_id'], row['condition'].lower()): row['eligible_for_distribution_metrics'] for row in technical_rows}
    for audit in candidate_audits:
        for representation in REPRESENTATIONS:
            if not eligibility[(audit['generator_id'], representation)]:
                continue
            for extractor_name in FEATURE_SPACES:
                records, summary = repeated_distribution_metrics(reference_features[('validation', extractor_name)], candidate_features[(audit['generator_id'], representation, extractor_name)], protocol, resampling_plan=shared_resampling_plan)
                distribution_repetitions.extend({'generator_id': audit['generator_id'], 'condition': representation.upper(), 'extractor': extractor_name, **record, 'real_ids': [validation_ids[index] for index in record['real_indices']], 'synthetic_ids': [candidate_ids[(audit['generator_id'], representation)][index] for index in record['synthetic_indices']]} for record in records)
                flat = {**summary['full_pool_distribution_estimates'], **summary['balanced_prdc_point_estimates'], **{f'{metric}_stability_{field}': value for metric, values in summary['stability_estimates'].items() for field, value in values.items()}}
                distribution_summaries.append({'generator_id': audit['generator_id'], 'condition': representation.upper(), 'extractor': extractor_name, **flat, 'full_pool_real_count': summary['full_pool_real_count'], 'full_pool_synthetic_count': summary['full_pool_synthetic_count'], 'balanced_prdc_point_real_count': summary['balanced_prdc_point_real_count'], 'balanced_prdc_point_synthetic_count': summary['balanced_prdc_point_synthetic_count'], 'stability_subset_size': summary['stability_subset_size'], 'stability_interval_type': summary['stability_interval_type'], 'full_pool_distribution_policy': summary['full_pool_distribution_policy'], 'fid_full_pool_caveat': summary['fid_full_pool_caveat']})
    write_csv_rows(OUTPUT_ROOT / 'distribution_metrics_repetitions.csv', distribution_repetitions)
    write_csv_rows(OUTPUT_ROOT / 'distribution_metrics_summary.csv', distribution_summaries)
distribution_summaries if distribution_summaries is not None else {'status': 'Deferred', 'resampling': protocol['resampling']}

## 9. Diversity analysis

In [ ]:
diversity_rows = None
if RUN_REAL_BENCHMARK:
    diversity_rows = list()
    for audit in candidate_audits:
        for representation in REPRESENTATIONS:
            key = (audit['generator_id'], representation)
            if key not in benchmark_keys:
                continue
            diversity_rows.append({'generator_id': audit['generator_id'], 'condition': representation.upper(),
                                   **diversity_metrics(candidate_paths[key], candidate_features[(*key, 'rad_dino')], pair_count=DIVERSITY_PAIR_COUNT, seed=protocol['sampling']['seed'])})
    write_csv_rows(OUTPUT_ROOT / 'diversity_metrics.csv', diversity_rows)
diversity_rows if diversity_rows is not None else 'Deferred; mandatory metrics are synthetic NN distance, perceptual-hash duplicate rate and deterministic MS-SSIM diversity. LPIPS is optional.'

## 10. Duplicate analysis

In [ ]:
synthetic_duplication_rows = None
if RUN_REAL_BENCHMARK:
    synthetic_duplication_rows = list()
    for audit in candidate_audits:
        for representation in REPRESENTATIONS:
            key = (audit['generator_id'], representation)
            if key not in benchmark_keys:
                continue
            path_map = dict(zip(candidate_ids[key], candidate_paths[key]))
            rows = build_synthetic_duplication_rows(candidate_features[(*key, 'rad_dino')], candidate_ids[key], path_map, protocol['memorization']['flag_rule'])
            synthetic_duplication_rows.extend({'generator_id': audit['generator_id'], 'condition': representation.upper(), **row} for row in rows)
    write_csv_rows(OUTPUT_ROOT / 'synthetic_duplication.csv', synthetic_duplication_rows)
synthetic_duplication_rows[:5] if synthetic_duplication_rows is not None else 'Deferred'

## 11. Train memorization analysis

The reference is the generator-specific complete declared training corpus. It includes every real negative, real positive, and positive augmentation actually seen by that generator; the validation-positive-only reference remains a separate descriptive analysis.

In [ ]:
train_memorization_rows = None
if RUN_REAL_BENCHMARK:
    train_memorization_rows = list(); training_corpus_by_generator = dict()
    for audit in candidate_audits:
        entry = next(item for item in registry['generators'] if item['id'] == audit['generator_id'])
        if not audit['eligible_for_benchmark_execution']:
            continue
        training_manifest = audit['training_corpus_manifest']
        train_paths, train_ids, train_labels, train_sources = training_corpus_from_manifest(ROOT, training_manifest)
        train_path_map = dict(zip(train_ids, map(Path, train_paths))); training_corpus_by_generator[audit['generator_id']] = training_manifest
        train_features, _ = get_or_extract_embeddings(embedding_cache_root / audit['generator_id'] / '_training_corpus' / 'rad_dino.npy', train_paths, train_ids, extractor='rad_dino', preprocessing='local recorded RAD-DINO processor', code_version='embedding-integrity-v4', source_manifest=str(ROOT / training_manifest), metadata_csv=str(ROOT / training_manifest), extractor_model_id=ENCODER_IDENTITIES['rad_dino']['model_repository'], extractor_weights_identifier=ENCODER_IDENTITIES['rad_dino']['identity_sha256'], extractor_identity=ENCODER_IDENTITIES['rad_dino'], extract_fn=lambda values, name: extract_features(values, name, allow_model_download=False))
        for representation in REPRESENTATIONS:
            key = (audit['generator_id'], representation)
            if key not in benchmark_keys:
                continue
            synthetic_path_map = dict(zip(candidate_ids[key], candidate_paths[key]))
            rows = build_train_memorization_rows(candidate_features[(*key, 'rad_dino')], train_features, candidate_ids[key], train_ids, synthetic_path_map, train_path_map, protocol['memorization']['flag_rule'], train_labels, train_sources)
            train_memorization_rows.extend({'generator_id': audit['generator_id'], 'condition': representation.upper(), **row} for row in rows)
    write_csv_rows(OUTPUT_ROOT / 'train_memorization.csv', train_memorization_rows)
train_memorization_rows[:5] if train_memorization_rows is not None else 'Deferred; only this table controls the memorization gate.'

## 12. Validation similarity analysis

In [ ]:
validation_similarity_rows = None
if RUN_REAL_BENCHMARK:
    validation_similarity_rows = list(); validation_path_map = dict(zip(validation_ids, map(Path, validation_paths)))
    for audit in candidate_audits:
        for representation in REPRESENTATIONS:
            key = (audit['generator_id'], representation)
            if key not in benchmark_keys:
                continue
            synthetic_path_map = dict(zip(candidate_ids[key], candidate_paths[key]))
            rows = build_validation_similarity_rows(candidate_features[(*key, 'rad_dino')], reference_features[('validation', 'rad_dino')], candidate_ids[key], validation_ids, synthetic_path_map, validation_path_map)
            validation_similarity_rows.extend({'generator_id': audit['generator_id'], 'condition': representation.upper(), **row} for row in rows)
    write_csv_rows(OUTPUT_ROOT / 'validation_similarity.csv', validation_similarity_rows)
validation_similarity_rows[:5] if validation_similarity_rows is not None else 'Deferred; this descriptive table never contains a memorization flag.'

## 13. Bootstrap/repeated subsampling

In [ ]:
if RUN_REAL_BENCHMARK:
    example_sizes = {row['generator_id']: evaluation_subset_size(row['representations']['filtered']['count'], real_reference_count, protocol['synthetic_pool_target']) for row in eligible}
    subsampling_status = {'evaluation_subset_size_by_generator': example_sizes, 'sampling': 'balanced without replacement', 'recorded_fields': ['repetition', 'seed', 'real_indices', 'synthetic_indices']}
else:
    subsampling_status = {'status': 'Deferred', 'sampling': 'balanced without replacement', 'seed': protocol['sampling']['seed']}
subsampling_status

## 14. Results tables

In [ ]:
generator_summary = generator_ranking = None
if RUN_REAL_BENCHMARK:
    generator_summary = list(); audits_by_id = {row['generator_id']: row for row in candidate_audits}
    distributions = {(row['generator_id'], row['condition'], row['extractor']): row for row in distribution_summaries}
    diversities = {(row['generator_id'], row['condition']): row for row in diversity_rows}
    for technical in technical_rows:
        generator_id, condition = technical['generator_id'], technical['condition']; audit = audits_by_id[generator_id]
        rad = distributions.get((generator_id, condition, 'rad_dino')); inception = distributions.get((generator_id, condition, 'inception_v3')); diversity = diversities.get((generator_id, condition), dict())
        train_group = [row for row in train_memorization_rows if row['generator_id'] == generator_id and row['condition'] == condition]
        validation_group = [row for row in validation_similarity_rows if row['generator_id'] == generator_id and row['condition'] == condition]
        duplication_group = [row for row in synthetic_duplication_rows if row['generator_id'] == generator_id and row['condition'] == condition]
        train_rate = sum(row['memorization_flag'] for row in train_group) / len(train_group) if train_group else None
        duplicate_rate = sum(row['duplicate_flag'] for row in duplication_group) / len(duplication_group) if duplication_group else None
        entry = next(item for item in registry['generators'] if item['id'] == generator_id)
        summary = {'generator_id': generator_id, 'condition': condition, 'family': audit['scientific_family'], 'role': audit['candidate_role'],
                   'eligible_for_selection': audit['eligible_for_downstream_selection'], 'technical_validity': technical['eligible_for_distribution_metrics'],
                   'technical_validity_rate': technical['technical_validity_rate'], 'filter_acceptance_rate': audit.get('filter_acceptance_rate'), 'valid_positive_images': technical['n_unique_valid_content'],
                   'provenance_manifest_valid': audit['provenance_manifest_valid'], 'lineage_complete': audit['lineage_complete'], 'filter_manifest_valid': audit['filter_manifest_valid'], 'filter_provenance_complete': audit.get('filter_provenance_complete'), 'n_corrupt': technical['n_corrupt'],
                   'training_corpus_manifest': audit.get('training_corpus_manifest'), 'training_corpus_manifest_valid': audit['training_corpus_manifest_valid'],
                   'raddino_kid': rad.get('kid_full_pool') if rad else None, 'raddino_kid_stability_low': rad.get('kid_stability_percentile_2_5') if rad else None,
                   'raddino_kid_stability_high': rad.get('kid_stability_percentile_97_5') if rad else None, 'raddino_kid_std': rad.get('kid_stability_standard_deviation') if rad else None,
                   'raddino_precision': rad.get('precision_balanced_point') if rad else None, 'raddino_recall': rad.get('recall_balanced_point') if rad else None,
                   'raddino_density': rad.get('density_balanced_point') if rad else None, 'raddino_coverage': rad.get('coverage_balanced_point') if rad else None,
                   'raddino_fid': rad.get('fid_full_pool') if rad else None, 'inception_kid': inception.get('kid_full_pool') if inception else None,
                   'inception_fid': inception.get('fid_full_pool') if inception else None, 'stability_interval_type': rad.get('stability_interval_type') if rad else None, 'ms_ssim_diversity': diversity.get('ms_ssim_diversity'),
                   'synthetic_duplicate_rate': duplicate_rate, 'synthetic_exact_duplicate_rate': diversity.get('synthetic_exact_duplicate_rate'),
                   'perceptual_hash_duplicate_rate': diversity.get('perceptual_hash_duplicate_rate'), 'train_memorization_rate': train_rate,
                   'validation_nearest_neighbour_distance': float(np.mean([row['embedding_distance'] for row in validation_group])) if validation_group else None,
                   **efficiency_from_manifest(ROOT, entry), 'metrics_complete': rad is not None and inception is not None, 'test_access': False}
        summary['technical_gates_passed'] = technical['eligible_for_distribution_metrics'] and not eligibility_failures(summary, protocol['eligibility_gates'])
        generator_summary.append(summary)
    write_csv_rows(OUTPUT_ROOT / 'generator_summary.csv', generator_summary)
    generator_ranking = list()
    filtered_rows = [row for row in generator_summary if row['condition'] == 'FILTERED']
    for family in ('finetuned', 'from_scratch'):
        generator_ranking.extend(rank_generator_family(filtered_rows, family, protocol['eligibility_gates']))
    write_csv_rows(OUTPUT_ROOT / 'generator_ranking.csv', generator_ranking)
    paired_rows = list()
    for family in ('finetuned', 'from_scratch'):
        top = [row for row in generator_ranking if row['family'] == family and row['eligible']][:2]
        if len(top) == 2:
            left = [row for row in distribution_repetitions if row['generator_id'] == top[0]['generator_id'] and row['condition'] == 'FILTERED' and row['extractor'] == 'rad_dino']
            right = [row for row in distribution_repetitions if row['generator_id'] == top[1]['generator_id'] and row['condition'] == 'FILTERED' and row['extractor'] == 'rad_dino']
            paired_rows.append({'family': family, **{key: value for key, value in paired_kid_differences(left, right, top[0]['generator_id'], top[1]['generator_id']).items() if key != 'paired_differences'}})
    if paired_rows: write_csv_rows(OUTPUT_ROOT / 'paired_generator_differences.csv', paired_rows)
    figure = plot_generator_summary(generator_summary); (OUTPUT_ROOT / 'figures').mkdir(parents=True, exist_ok=True); figure.savefig(OUTPUT_ROOT / 'figures/generator_summary.png', dpi=150)
generator_summary if generator_summary is not None else {'status': 'Deferred; no synthetic result rows were generated', 'output_root': str(OUTPUT_ROOT)}

## 15. Pareto analysis

In [ ]:
ranking_view = generator_ranking if generator_ranking is not None else 'Deferred; ranking order is eligibility → RAD-DINO KID → coverage → precision → RAD-DINO FID → Inception KID → stability → generator_id.'
ranking_view

## 16. Visual panels

In [ ]:
panel_outputs = None
if RUN_REAL_BENCHMARK:
    panel_outputs = list(); panel_root = OUTPUT_ROOT / 'diagnostic_panels'
    for audit in candidate_audits:
        for representation in REPRESENTATIONS:
            key = (audit['generator_id'], representation)
            if key not in benchmark_keys:
                continue
            condition = representation.upper(); synthetic_map = dict(zip(candidate_ids[key], candidate_paths[key]))
            train_panel = [{**row, 'source_id': row['nearest_train_id']} for row in train_memorization_rows if row['generator_id'] == audit['generator_id'] and row['condition'] == condition]
            validation_panel = [{**row, 'source_id': row['nearest_validation_id']} for row in validation_similarity_rows if row['generator_id'] == audit['generator_id'] and row['condition'] == condition]
            synthetic_panel = [{**row, 'source_id': row['nearest_synthetic_id']} for row in synthetic_duplication_rows if row['generator_id'] == audit['generator_id'] and row['condition'] == condition]
            for name, rows, references in (('train', train_panel, train_path_map), ('validation', validation_panel, validation_path_map), ('synthetic', synthetic_panel, synthetic_map)):
                output = panel_root / f"{audit['generator_id']}_{representation}_{name}.png"
                panel_outputs.append(str(render_similarity_panel(rows, synthetic_map, references, output, f"{audit['generator_id']} {condition}: synthetic | nearest {name}")))
panel_outputs if panel_outputs is not None else {'status': 'Deferred', 'policy': 'deterministic closest / median / farthest for train, validation and synthetic'}

## 17. Family-specific conclusions

Conclusions are written only after the result table exists. The 50-step Stable Diffusion row is a sampling ablation beside its 100-step counterpart, not an independent eligible winner. The first LDM remains a descriptive historical baseline until lineage is demonstrated. FID is secondary and explicitly unstable at the small real-reference count.